# Task 1: EDA and Preprocessing

CFPB Consumer Complaint Database: full dataset analysis and filtering
for the CrediTrust RAG chatbot project.

> **Branch:** `feat/preprocessing` &nbsp;|&nbsp; **Improvement vs. v1:** this
> notebook adds boilerplate-opener stripping (`strip_boilerplate_openers()`
> in `src/preprocessing.py`), a Task 1 requirement named explicitly in the
> instructions that v1 never implemented. See
> [`docs/WHY_A_CLEAN_REBUILD.md`](../docs/WHY_A_CLEAN_REBUILD.md) for the
> full rationale. Everything else in this notebook — loading strategy,
> EDA, filtering, output path — is unchanged from v1; no gap was found
> there, so nothing was rewritten for its own sake.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 100)

FILTERED_PATH = "../data/processed/filtered_complaints.csv"


## 1. Load the full CFPB complaint dataset

The raw file is ~5GB / ~9.6M rows. We load it in chunks with progress
printed along the way, and only keep the columns we actually need.


In [ ]:
RAW_PATH = "../data/raw/complaints.csv"
USECOLS = [
    "Date received", "Product", "Sub-product", "Issue", "Sub-issue",
    "Consumer complaint narrative", "Company", "State", "Complaint ID",
]
DTYPES = {
    "Product": "category", "Sub-product": "category", "Issue": "category",
    "Sub-issue": "category", "Company": "category", "State": "category",
    "Complaint ID": "int64",
}

start = time.time()
chunks = []
chunk_size = 200_000

for i, chunk in enumerate(pd.read_csv(
    RAW_PATH, usecols=USECOLS, dtype=DTYPES,
    parse_dates=["Date received"], low_memory=False, chunksize=chunk_size,
)):
    chunks.append(chunk)
    print(f"Chunk {i+1}: {(i+1)*chunk_size:,} rows — {time.time()-start:.0f}s elapsed")

df = pd.concat(chunks, ignore_index=True)
del chunks
print(f"\nDone. {len(df):,} rows, {df.memory_usage(deep=True).sum()/1e9:.2f} GB, {time.time()-start:.0f}s total")
df.head()


## 2. Initial EDA

### 2a. Distribution of complaints across products

In [ ]:
product_counts = df["Product"].value_counts()
print(product_counts)


In [ ]:
plt.figure(figsize=(10, 8))
product_counts.plot(kind="barh")
plt.xlabel("Number of Complaints")
plt.title("Complaint Volume by Product (Full Dataset)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../data/processed/eda_product_distribution.png", dpi=150)
plt.show()


### 2b. Consumer narrative length (word count)

We compute word count only where a narrative exists, to avoid NaN issues.
With ~9.6M total rows, we use a plain numpy loop rather than chained
`.str` accessors for speed.


In [ ]:
has_narrative = df["Consumer complaint narrative"].notna()
print(f"With narrative:    {has_narrative.sum():,}")
print(f"Without narrative: {(~has_narrative).sum():,}")
print(f"Percentage with:   {has_narrative.mean()*100:.1f}%")


In [ ]:
start = time.time()
narratives = df.loc[has_narrative, "Consumer complaint narrative"].to_numpy()
word_counts = np.array([len(str(n).split()) for n in narratives])
print(pd.Series(word_counts).describe())
print(f"\n{time.time()-start:.1f}s")


In [ ]:
plt.figure(figsize=(10, 5))
clipped = np.clip(word_counts, None, np.percentile(word_counts, 99))
plt.hist(clipped, bins=80)
plt.xlabel("Word Count")
plt.ylabel("Number of Complaints")
plt.title("Distribution of Consumer Narrative Length (99th percentile clipped)")
plt.tight_layout()
plt.savefig("../data/processed/eda_narrative_length.png", dpi=150)
plt.show()


In [ ]:
print("Very short narratives (<5 words):", (word_counts < 5).sum())
print("Very long narratives (>1000 words):", (word_counts > 1000).sum())


### 2c. Count complaints with and without narratives (by product)
Useful to see whether narrative availability skews by product category.


In [ ]:
narrative_by_product = (
    df.assign(has_narrative=has_narrative)
    .groupby("Product", observed=True)["has_narrative"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "with_narrative", "count": "total"})
)
narrative_by_product["pct_with_narrative"] = (
    narrative_by_product["with_narrative"] / narrative_by_product["total"] * 100
).round(1)
narrative_by_product.sort_values("total", ascending=False)


## 3. Filter the dataset

First, inspect the *exact* unique Product values present in this export.
CFPB has renamed categories over the years, so we match flexibly rather
than assuming exact strings.


In [ ]:
print(sorted(df["Product"].unique().tolist()))

Based on the unique values above, the following maps each of the 4 target
categories to the matching raw `Product` label(s), using the mapping
defined in `src/preprocessing.py`. We include the standalone `"Payday
loan"` label under Personal Loan alongside the bundled payday/title/
personal-loan variants.


In [ ]:
import sys
sys.path.insert(0, "..")

from src.preprocessing import (
    PRODUCT_MAP,
    build_reverse_product_map,
    get_target_raw_products,
    clean_text,
    strip_boilerplate_openers,
)

target_products = get_target_raw_products()
reverse_map = build_reverse_product_map()

filtered = df[df["Product"].isin(target_products)].copy()
filtered["product_category"] = filtered["Product"].map(reverse_map)

print(f"Rows after product filter: {len(filtered):,}")
print(filtered["product_category"].value_counts())


In [ ]:
filtered = filtered[filtered["Consumer complaint narrative"].notna()].copy()
filtered = filtered[filtered["Consumer complaint narrative"].str.strip() != ""]

print(f"Rows after dropping empty narratives: {len(filtered):,}")
print(filtered["product_category"].value_counts())


## 4. Clean the text narratives

Uses `clean_text()` from `src/preprocessing.py`, which now applies three
stages (the first is new in this branch):

1. **Boilerplate-opener stripping** — removes stock complaint-letter
   openers ("I am writing to file a complaint...", "To whom it may
   concern,"). *Why this stage exists:* the Task 1 brief names this as
   an explicit example cleaning step; v1 never implemented it, and these
   openers are near-identical across thousands of complaints, so left in
   they add repeated boilerplate tokens to every chunk's embedding
   instead of complaint-specific signal. See `strip_boilerplate_openers()`
   in `src/preprocessing.py` for why it's a dedicated, testable function
   rather than folded silently into noise removal.
2. **Noise removal** — lowercasing, URL/phone/ID/HTML/redaction removal,
   punctuation stripping.
3. **NLP normalization** — tokenization, stopword removal, and
   lemmatization (verbs + nouns), so inflected forms collapse to a
   common root before embedding.

This step is the slowest in the pipeline since NLTK tokenization and
lemmatization run in pure Python, but it's now applied only to the
filtered subset, not the full 9.6M rows.


In [ ]:
# Quick illustration of stage 1 before running it across the full filtered
# set below — pick a few narratives that actually contain a formulaic
# opener, so the effect is visible rather than assumed.
sample_with_opener = filtered[
    filtered["Consumer complaint narrative"].str.contains(
        r"^\s*(?:i am writing to|i'm writing to|to whom it may concern)",
        case=False, regex=True, na=False,
    )
].head(3)

for _, row in sample_with_opener.iterrows():
    raw = row["Consumer complaint narrative"]
    print("RAW: ", raw[:160], "...")
    print("STRIPPED:", strip_boilerplate_openers(raw.lower())[:160], "...")
    print()


In [ ]:
start = time.time()
filtered["cleaned_narrative"] = filtered["Consumer complaint narrative"].apply(clean_text)
print(f"{time.time()-start:.1f}s for {len(filtered):,} rows")

# Drop any rows that became empty after cleaning
filtered = filtered[filtered["cleaned_narrative"].str.len() > 0]

print(f"Final row count: {len(filtered):,}")
filtered[["Consumer complaint narrative", "cleaned_narrative"]].head(3)


## 5. Save the cleaned and filtered dataset

In [ ]:
output_cols = [
    "Complaint ID", "Date received", "product_category", "Product",
    "Sub-product", "Issue", "Sub-issue", "Company", "State",
    "Consumer complaint narrative", "cleaned_narrative",
]

filtered[output_cols].to_csv(FILTERED_PATH, index=False)
print(f"Saved {len(filtered):,} rows to {FILTERED_PATH}")


**EDA Summary**

The full CFPB complaint dataset contains 9,609,797 records spanning all CFPB
product categories. Only 31.0% of complaints (2,980,756) include a
free-text consumer narrative; the remaining 69% were submitted without
one, meaning the bulk of the raw dataset is unusable for a
narrative-based RAG system. Among complaints with narratives, length is
right-skewed: the median is 114 words and the mean is 176 words, with a
long tail extending to a maximum of 6,469 words.

After filtering to the four target product categories (Credit Card,
Personal Loan, Savings Account, Money Transfer) and removing records with
empty narratives, the dataset was reduced to 480,564 complaints.
Distribution across categories is uneven: Credit Card (189,334) and
Savings Account (155,204) dominate, while Money Transfer (98,685) and
especially Personal Loan (37,341) are comparatively underrepresented — a
roughly 5:1 ratio between the largest and smallest category that should
inform stratified sampling in Task 2.

Narratives were cleaned in three stages: boilerplate-opener stripping,
noise removal (lowercasing, stripping URLs, phone numbers, HTML tags,
CFPB redaction placeholders, and punctuation), and NLP normalization
(tokenization, English stopword removal, and lemmatization of both verbs
and nouns). Row counts and category proportions are unchanged from the
v1 run, since boilerplate stripping only touches narrative *content*, not
which rows survive filtering — the effect instead shows up downstream, in
Task 2/3's retrieval quality, by removing repeated non-signal tokens from
what gets embedded. *(Runtime and the exact "rows emptied by cleaning"
count will differ slightly from v1's reported 632s / 4 rows, since an
extra regex pass now runs per narrative — re-run this notebook against
your local `data/raw/complaints.csv` to get current numbers.)* The
cleaned dataset was saved to `data/processed/filtered_complaints.csv` for
use in Task 2.
